In [ ]:
import tempfile, os, subprocess, torch
os.chdir(tempfile.mkdtemp())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
%%writefile worker.py
import os
import sys
from pathlib import Path
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(71)
torch.set_num_threads(1)
model = torch.nn.Linear(16, 4).to(device)
if sys.argv[1] == "short":
    with torch.inference_mode():
        model(torch.ones(1, 16, device=device)).cpu()
    print("Short inference job finished on", device, flush=True)
    sys.exit()

optimizer = torch.optim.SGD(model.parameters(), lr=.01, momentum=.9)
checkpoint = Path("checkpoint.pt")
start = 0
if sys.argv[1] == "resume":
    state = torch.load(checkpoint, map_location=device, weights_only=True)
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    start = state["step"]
    print("Resuming after step", start, flush=True)

for step in range(start + 1, 101):
    generator = torch.Generator().manual_seed(step)
    batch = torch.randn(32, 16, generator=generator).to(device)
    target = batch[:, :4] * 2
    optimizer.zero_grad()
    (model(batch) - target).square().mean().backward()
    optimizer.step()
    if step == 30 and sys.argv[1] == "interruptible":
        print("READY", flush=True)
        if input() == "CHECKPOINT":
            torch.save(dict(step=step, model=model.state_dict(), optimizer=optimizer.state_dict()), "pending.pt")
            os.replace("pending.pt", checkpoint)
            print("Checkpoint committed", flush=True)
            sys.exit(75)
torch.save(model.state_dict(), sys.argv[1] + ".pt")


In [ ]:
%%writefile scheduler.cpp
#include <cerrno>
#include <cstdio>
#include <cstdlib>
#include <string>
#include <sys/wait.h>
#include <unistd.h>

void check(bool ok) {
    if (!ok) { std::perror("scheduler"); std::exit(1); }
}

int run(const char* mode, bool interruptible = false) {
    int commands[2], events[2];
    check(pipe(commands) == 0 && pipe(events) == 0);
    pid_t pid = fork();
    check(pid >= 0);
    if (pid == 0) {
        if (dup2(commands[0], STDIN_FILENO) < 0 || dup2(events[1], STDOUT_FILENO) < 0) _exit(126);
        for (int fd : {commands[0], commands[1], events[0], events[1]}) close(fd);
        execlp("python3", "python3", "-u", "worker.py", mode, nullptr);
        _exit(127);
    }
    close(commands[0]);
    close(events[1]);
    FILE* output = fdopen(events[0], "r");
    check(output != nullptr);
    char line[512];
    while (fgets(line, sizeof(line), output)) {
        std::fputs(line, stdout);
        if (interruptible && std::string(line) == "READY\n") {
            std::puts("Short job arrived. Requesting checkpoint.");
            check(write(commands[1], "CHECKPOINT\n", 11) == 11);
        }
    }
    fclose(output);
    close(commands[1]);
    int status;
    while (waitpid(pid, &status, 0) == -1) check(errno == EINTR);
    return WIFEXITED(status) ? WEXITSTATUS(status) : 1;
}

int main() {
    if (run("interruptible", true) != 75) return 1;
    if (run("short") != 0) return 1;
    return run("resume");
}


In [ ]:
subprocess.run(['g++','-std=c++17','-Wall','scheduler.cpp','-o','scheduler'], check=True)
subprocess.run(['./scheduler'], check=True)

In [ ]:
subprocess.run(['python3', 'worker.py', 'reference'], check=True)
def weights(name):
    return torch.load(name + '.pt', map_location='cpu', weights_only=True)
reference, resumed = weights('reference'), weights('resume')
for key in reference:
    torch.testing.assert_close(reference[key], resumed[key])
print('PASS: resumed weights match the uninterrupted reference within tolerance.')